# Employee Retention & Churn Prediction

## Machine Learning Notebook

### Objective:
Develop a machine learning classification model to predict
whether an employee is likely to churn within one year.

### Workflow:
- Load cleaned analytical data
- Understand the prediction dataset
- Define target variable
- Select predictive features
- Perform train-test split
- Preprocess numerical and categorical features
- Train baseline model
- Train tree-based model
- Evaluate model performance
- Compare models
- Analyze feature importance
- Save the best-performing model

1. Import Libraries

2. Load Data

3. Data Overview
   - Shape
   - Columns
   - Data types
   - Missing values
   - Duplicate records

4. Target Variable Analysis
   - churned_within_1yr distribution
   - Class balance

5. Feature Selection
   - Define X
   - Define y
   - Remove identifiers
   - Check for leakage

6. Train-Test Split

7. Data Preprocessing
   - Numerical features
   - Categorical features
   - Imputation
   - Scaling
   - One-hot encoding

8. Baseline Model
   - Logistic Regression

9. Machine Learning Models

10. Model Evaluation
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - ROC-AUC
   - Confusion Matrix

11. Model Comparison

12. Feature Importance

13. Business Interpretation

14. Save Best Model

In [1]:
# Data Manipulation & Numerical Computing
import pandas as pd
import numpy as np
import scipy.sparse as sp

# Data Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Train-Test Split
from sklearn.model_selection import train_test_split

# Data Preprocessing
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Classification Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
)
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

 # Model Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

In [2]:
jobs = pd.read_csv(r"C:\Users\babus\Data_Spark\Project\Employee_Retention_Churn_Analysis\Deployment\data\processed\job_listings.csv")
recruitment = pd.read_csv(r"C:\Users\babus\Data_Spark\Project\Employee_Retention_Churn_Analysis\Deployment\data\processed\recruiting_kpis.csv")
retention = pd.read_csv(r"C:\Users\babus\Data_Spark\Project\Employee_Retention_Churn_Analysis\Deployment\data\processed\retention_kpis.csv")

In [3]:
retention.shape , recruitment.shape , jobs.shape

((62426, 4), (250000, 5), (250000, 15))

In [4]:
merged_df = retention.merge(recruitment,on='job_id',how="left")
final_df = merged_df.merge(jobs,on='job_id',how="left")
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 62426 entries, 0 to 62425
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   job_id                62426 non-null  int64  
 1   starting_salary       62426 non-null  float64
 2   tenure_months         62426 non-null  int64  
 3   churned_within_1yr    62426 non-null  bool   
 4   num_applicants        62426 non-null  int64  
 5   avg_response_hours    62426 non-null  str    
 6   offer_extended        62426 non-null  bool   
 7   offer_accepted        62426 non-null  bool   
 8   title                 62426 non-null  str    
 9   industry              62426 non-null  str    
 10  company_size          62426 non-null  str    
 11  company_tier          62426 non-null  str    
 12  country               62426 non-null  str    
 13  city                  62426 non-null  str    
 14  remote_policy         62426 non-null  str    
 15  experience_level      62426 no

In [5]:
for col in final_df:
    print(f"{col}")
    print(final_df[col].unique())

job_id
[     2     10     14 ... 249979 249987 249990]
starting_salary
[ 79095.83  98917.21  87016.68 ...  75867.43 100725.03 102310.8 ]
tenure_months
[23 18 10 11  9 13 14  4  3 22 12 20  7  8  2 24 15  0 21  6 17  5 16  1
 19]
churned_within_1yr
[False  True]
num_applicants
[298 167 379 330 234  64 310 156 170  35 495 469 419 400 255 232 373 236
 212 168  72 104 328 247 191 466 411 124 442 311 271 147 447 266 285  22
 343 208 145 385 346 491 300 324 376 258 254 325 204 398 202 201  96 117
 341  53 127 164 125  97  52 182 478 482 334  59  68  20 449  24 320 484
 225 150 181 404 277 261 229 139 106 245 367  16 119  51  50 129  60 397
 414 437 477  76 391  56  65 383 375 342 444 132 415  69 432 278 450 436
 210 165 224 286 429 276  80 189 401  45 370 242 347 184 359 207 344 282
 262 454 453 390 137 237 238  58 246  17 209 461 121   9  40 316 417 293
 307 327  74 463 183 475 479  48 396 348 256 120 148 362 456  19 149 180
 227  66  14 111 445 173  18 439  47 171 473 470 468 223 358 305 1

In [6]:
final_df.describe()

,job_id,starting_salary,tenure_months,num_applicants
count,62426.000000,62426.000000,62426.000000,62426.000000
mean,124988.152773,84969.196038,11.995034,252.715631
std,72091.924806,14975.387678,7.208699,143.464220
min,2.000000,18055.470000,0.000000,5.000000
25%,62736.500000,74882.787500,6.000000,128.000000
50%,124797.500000,85008.330000,12.000000,253.000000
75%,187301.500000,95025.595000,18.000000,377.000000
max,249990.000000,144653.020000,24.000000,500.000000


## Feature Selection

In [7]:
features = [
    "starting_salary",
    "title",
    "industry",
    "company_size",
    "company_tier",
    "remote_policy",
    "experience_level",
    "employment_type"
]

In [8]:
X = final_df[features].copy()
Y = final_df['churned_within_1yr'].copy()

In [9]:
print(type(X))
print(X.columns.tolist())

<class 'pandas.DataFrame'>
['starting_salary', 'title', 'industry', 'company_size', 'company_tier', 'remote_policy', 'experience_level', 'employment_type']


In [10]:
X.shape , Y.shape

((62426, 8), (62426,))

In [11]:
X['starting_salary'] = X['starting_salary'].astype(int)

In [12]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 62426 entries, 0 to 62425
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   starting_salary   62426 non-null  int64
 1   title             62426 non-null  str  
 2   industry          62426 non-null  str  
 3   company_size      62426 non-null  str  
 4   company_tier      62426 non-null  str  
 5   remote_policy     62426 non-null  str  
 6   experience_level  62426 non-null  str  
 7   employment_type   62426 non-null  str  
dtypes: int64(1), str(7)
memory usage: 3.8 MB


In [13]:
for col in X:
    print(f"{col}")
    print(X[col].unique())

starting_salary
[ 79095  98917  87016 ...  79091  69570 100725]
title
<StringArray>
[    'AI Researcher',  'Business Analyst',   'Cloud Architect',
     'Data Engineer',    'Data Scientist',   'DevOps Engineer',
   'Product Manager',       'QA Engineer', 'Software Engineer',
       'UX Designer']
Length: 10, dtype: str
industry
<StringArray>
[     'Education',         'Retail',     'Technology',        'Finance',
  'Manufacturing', 'Transportation',     'Healthcare']
Length: 7, dtype: str
company_size
<StringArray>
['501-1000', '1001-5000', '201-500', '51-200', '5001-10000', '10000+', '1-50']
Length: 7, dtype: str
company_tier
<StringArray>
['Enterprise', 'Corporation', 'Mid-Market', 'Scale-up', 'MNC', 'Startup']
Length: 6, dtype: str
remote_policy
<StringArray>
['Remote', 'Onsite', 'Hybrid']
Length: 3, dtype: str
experience_level
<StringArray>
['Senior', 'Director', 'Entry', 'Mid']
Length: 4, dtype: str
employment_type
<StringArray>
['Internship', 'Contract', 'Full-time', 'Part-time']

## Data Preprocessing

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X,Y,test_size=0.2,random_state=1,stratify=Y)

In [15]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((49940, 8), (12486, 8), (49940,), (12486,))

In [16]:
y_train = y_train.astype(int)
y_test = y_test.astype(int)

In [17]:
X_train.select_dtypes(include='str').columns

Index(['title', 'industry', 'company_size', 'company_tier', 'remote_policy',
       'experience_level', 'employment_type'],
      dtype='str')

In [18]:
X_train[['company_size', 'experience_level', 'company_tier']].isna().sum()

company_size        0
experience_level    0
company_tier        0
dtype: int64

In [19]:
company_size_order = ['1-50','51-200','201-500','501-1000','1001-5000','5001-10000','10000+']
experience_level_order = ['Entry','Mid','Senior','Director']
company_tier_order = ['Startup', 'Scale-up', 'Mid-Market', 'Enterprise', 'MNC', 'Corporation']

ordinal_encoder = OrdinalEncoder(
    categories=[company_size_order,experience_level_order,company_tier_order],
    handle_unknown='use_encoded_value',
    unknown_value=-1,
    encoded_missing_value=-1,
)

one_hot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
Standard_Scaler = StandardScaler()

preprocessor = ColumnTransformer(
    transformers = [
        (
            'ordinal',
             ordinal_encoder,
             ['company_size','experience_level','company_tier']
        ),
        (
            'nominal', 
            one_hot_encoder, 
            ['title', 'industry', 'remote_policy', 'employment_type']
        ),
        (
            'numeric', 
         Standard_Scaler, 
         ['starting_salary']
        )
    ],
    remainder='drop' ,
    verbose_feature_names_out=False
)


In [20]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [21]:
X_train

array([[ 4.        ,  1.        ,  5.        , ...,  1.        ,
         0.        ,  0.38542662],
       [ 4.        ,  2.        ,  5.        , ...,  0.        ,
         1.        , -0.04625613],
       [ 2.        ,  2.        ,  2.        , ...,  0.        ,
         0.        ,  2.20996128],
       ...,
       [ 5.        ,  0.        ,  4.        , ...,  0.        ,
         0.        ,  0.58310065],
       [ 6.        ,  3.        ,  4.        , ...,  0.        ,
         0.        ,  0.73637303],
       [ 5.        ,  3.        ,  4.        , ...,  0.        ,
         0.        ,  0.672904  ]], shape=(49940, 28))

## Machine Learning Models

In [22]:
Logistic_Regression = LogisticRegression()
KNeighbors_Classifier = KNeighborsClassifier()
DecisionTree_Classifier = DecisionTreeClassifier()
RandomForest_Classifier = RandomForestClassifier()
ExtraTrees_Classifier = ExtraTreesClassifier()
GradientBoosting_Classifier = GradientBoostingClassifier()
HistGradientBoosting_Classifier = HistGradientBoostingClassifier()
Linear_SVC = LinearSVC()
Gaussian_NB = GaussianNB()
XGB_Classifier = XGBClassifier()

In [23]:
models = {
    "Logistic Regression": Logistic_Regression,
    "KNN": KNeighbors_Classifier,
    "Decision Tree": DecisionTree_Classifier,
    "Random Forest": RandomForest_Classifier,
    "Extra Trees": ExtraTrees_Classifier,
    "Gradient Boosting": GradientBoosting_Classifier,
    "HistGradient Boosting": HistGradientBoosting_Classifier,
    "Linear SVC": Linear_SVC,
    "Gaussian NB": Gaussian_NB,
    "XGBoost": XGB_Classifier
}

trained_models = {}

for name, model in models.items():
    try:
        print(f"Training {name}...")

        model.fit(X_train, Y_train)

        trained_models[name] = model

        print(f"{name} trained successfully")

    except Exception as e:
        print(f"{name} failed")
        print(f"Error: {e}")

    print("-" * 60)

Training Logistic Regression...
Logistic Regression failed
Error: name 'Y_train' is not defined
------------------------------------------------------------
Training KNN...
KNN failed
Error: name 'Y_train' is not defined
------------------------------------------------------------
Training Decision Tree...
Decision Tree failed
Error: name 'Y_train' is not defined
------------------------------------------------------------
Training Random Forest...
Random Forest failed
Error: name 'Y_train' is not defined
------------------------------------------------------------
Training Extra Trees...
Extra Trees failed
Error: name 'Y_train' is not defined
------------------------------------------------------------
Training Gradient Boosting...
Gradient Boosting failed
Error: name 'Y_train' is not defined
------------------------------------------------------------
Training HistGradient Boosting...
HistGradient Boosting failed
Error: name 'Y_train' is not defined
----------------------------------

In [24]:

results = []

def evaluate_model(model, X_train, X_test, y_train, y_test, name):
    # --- CONVERT TO DENSE TO FIX HISTGRADIENTBOOSTING ---
    if sp.issparse(X_train):
        X_train = X_train.toarray()
        X_test = X_test.toarray()
    elif hasattr(X_train, "to_numpy"):
        try:
            X_train = np.asarray(X_train)
            X_test = np.asarray(X_test)
        except:
            pass

    # 1. Fit the model once
    model.fit(X_train, y_train)
    
    # 2. Get predictions
    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)

    # 3. Get probability scores or decision boundary metrics
    if hasattr(model, "predict_proba"):
        y_train_score = model.predict_proba(X_train)[:, 1]
        y_test_score = model.predict_proba(X_test)[:, 1]
    else:
        y_train_score = model.decision_function(X_train)
        y_test_score = model.decision_function(X_test)
        
    return {
        "Model": name,
        
        "Train Accuracy": accuracy_score(y_train, y_train_pred),
        "Test Accuracy": accuracy_score(y_test, y_test_pred),

        "Train Precision": precision_score(y_train, y_train_pred),
        "Test Precision": precision_score(y_test, y_test_pred),

        "Train Recall": recall_score(y_train, y_train_pred),
        "Test Recall": recall_score(y_test, y_test_pred),

        "Train F1 Score": f1_score(y_train, y_train_pred),
        "Test F1 Score": f1_score(y_test, y_test_pred),

        "Train ROC-AUC": roc_auc_score(y_train, y_train_score),
        "Test ROC-AUC": roc_auc_score(y_test, y_test_score)
    }
    
# Your 10 model execution lines remain exactly the same
results.append(evaluate_model(Logistic_Regression, X_train, X_test, y_train, y_test, "Logistic Regression"))
results.append(evaluate_model(KNeighbors_Classifier, X_train, X_test, y_train, y_test, "KNN"))
results.append(evaluate_model(DecisionTree_Classifier, X_train, X_test, y_train, y_test, "Decision Tree"))
results.append(evaluate_model(RandomForest_Classifier, X_train, X_test, y_train, y_test, "Random Forest"))
results.append(evaluate_model(ExtraTrees_Classifier, X_train, X_test, y_train, y_test, "Extra Trees"))
results.append(evaluate_model(GradientBoosting_Classifier, X_train, X_test, y_train, y_test, "Gradient Boosting"))
results.append(evaluate_model(HistGradientBoosting_Classifier, X_train, X_test, y_train, y_test, "HistGradient Boosting"))
results.append(evaluate_model(Linear_SVC, X_train, X_test, y_train, y_test, "Linear SVC"))
results.append(evaluate_model(Gaussian_NB, X_train, X_test, y_train, y_test, "Gaussian NB"))
results.append(evaluate_model(XGB_Classifier, X_train, X_test, y_train, y_test, "XGBoost"))

reslt_table = pd.DataFrame(results)
reslt_table


,Model,Train Accuracy,Test Accuracy,Train Precision,Test Precision,Train Recall,Test Recall,Train F1 Score,Test F1 Score,Train ROC-AUC,Test ROC-AUC
0,Logistic Regression,0.519483,0.515617,0.517223,0.465517,0.039318,0.035904,0.073081,0.066667,0.511159,0.501277
1,KNN,0.683540,0.497517,0.677900,0.477774,0.653782,0.460938,0.665623,0.469205,0.742116,0.500061
2,Decision Tree,0.999980,0.500080,1.000000,0.481198,0.999958,0.480718,0.999979,0.480958,1.000000,0.499361
3,Random Forest,0.999980,0.503604,0.999958,0.484234,1.000000,0.464594,0.999979,0.474211,1.000000,0.499058
4,Extra Trees,0.999980,0.501682,1.000000,0.482970,0.999958,0.485539,0.999979,0.484251,1.000000,0.501192
5,Gradient Boosting,0.535663,0.514336,0.605576,0.475758,0.103824,0.078291,0.177257,0.134456,0.557880,0.500282
6,HistGradient Boosting,0.530697,0.514336,0.648122,0.454887,0.056650,0.040226,0.104193,0.073916,0.557803,0.497225
7,Linear SVC,0.519283,0.516098,0.514473,0.471861,0.039152,0.036237,0.072767,0.067305,0.511162,0.501402
8,Gaussian NB,0.512074,0.505446,0.490589,0.480614,0.332585,0.327626,0.396423,0.389641,0.510044,0.499404
9,XGBoost,0.681778,0.508970,0.698156,0.488714,0.598047,0.413896,0.644236,0.448204,0.754810,0.503828


In [26]:
final_df["churned_within_1yr"].value_counts()

churned_within_1yr
False    32350
True     30076
Name: count, dtype: int64

In [27]:
final_df["churned_within_1yr"].value_counts(normalize=True) * 100

churned_within_1yr
False    51.821356
True     48.178644
Name: proportion, dtype: float64

In [28]:
categorical_cols = [
    "title",
    "industry",
    "company_size",
    "company_tier",
    "remote_policy",
    "experience_level",
    "employment_type"
]

for col in categorical_cols:
    print(f"\n{'='*60}")
    print(f"{col}")
    print(pd.crosstab(
        final_df[col],
        final_df["churned_within_1yr"],
        normalize="index"
    ).round(3))


title
churned_within_1yr  False  True 
title                           
AI Researcher       0.529  0.471
Business Analyst    0.510  0.490
Cloud Architect     0.523  0.477
Data Engineer       0.514  0.486
Data Scientist      0.524  0.476
DevOps Engineer     0.523  0.477
Product Manager     0.507  0.493
QA Engineer         0.522  0.478
Software Engineer   0.515  0.485
UX Designer         0.514  0.486

industry
churned_within_1yr  False  True 
industry                        
Education           0.519  0.481
Finance             0.519  0.481
Healthcare          0.517  0.483
Manufacturing       0.516  0.484
Retail              0.518  0.482
Technology          0.521  0.479
Transportation      0.517  0.483

company_size
churned_within_1yr  False  True 
company_size                    
1-50                0.520  0.480
10000+              0.514  0.486
1001-5000           0.526  0.474
201-500             0.518  0.482
5001-10000          0.510  0.490
501-1000            0.513  0.487
51-200      

In [29]:
final_df[
    ["starting_salary", "tenure_months", "churned_within_1yr"]
].corr()

,starting_salary,tenure_months,churned_within_1yr
starting_salary,1.000000,-0.001682,0.002515
tenure_months,-0.001682,1.000000,-0.864849
churned_within_1yr,0.002515,-0.864849,1.000000


In [31]:
from scipy.stats import chi2_contingency

for col in categorical_cols:

    table = pd.crosstab(
        final_df[col],
        final_df["churned_within_1yr"]
    )

    chi2, p_value, dof, expected = chi2_contingency(table)

    print(f"{col:20} p-value = {p_value:.6f}")

title                p-value = 0.295625
industry             p-value = 0.995753
company_size         p-value = 0.145359
company_tier         p-value = 0.098194
remote_policy        p-value = 0.203216
experience_level     p-value = 0.647385
employment_type      p-value = 0.877904


## Modeling Conclusion

Multiple classification algorithms were evaluated to predict `churned_within_1yr`, including Logistic Regression, KNN, Decision Tree, Random Forest, Extra Trees, Gradient Boosting, HistGradient Boosting, Linear SVC, Gaussian Naive Bayes, and XGBoost.

The initial models produced test ROC-AUC scores close to 0.50 when `tenure_months` was excluded, indicating that the selected features did not contain sufficient predictive information to reliably distinguish between employees who churned and those who did not.

Further investigation identified a significant data leakage issue with `tenure_months`. Employees with tenure below 12 months were consistently classified as having churned within one year, while employees with tenure of 12 months or more were consistently classified as non-churned. Including `tenure_months` therefore resulted in artificially perfect model performance, with several models achieving 100% accuracy and ROC-AUC. This performance cannot be considered valid because the feature effectively contains the target definition.

Additional analysis of the remaining categorical variables showed very similar churn rates across categories, and the statistical tests provided little evidence of meaningful relationships between these variables and the target.

Therefore, no classification model was selected for deployment from the current dataset. Hyperparameter tuning was also not performed because the baseline models did not demonstrate meaningful predictive capability without the leaked feature.

### Final Finding

The main limitation was not the choice of classification algorithm but the quality and construction of the target variable. The current dataset does not provide sufficient legitimate predictive information for `churned_within_1yr`.

For a future version of the project, the target-generation logic should be redesigned so that employee churn is influenced by meaningful business factors such as compensation, role, experience level, employment type, company characteristics, and work arrangement, while avoiding features that directly define the target.
